# Project Aura 2.0 - Notebook 05: Gesture RecognitionThis notebook demonstrates gesture recognition:- Detect hand gestures using `src.agents.gesture`- MediaPipe Hands integration (when available)- Lightweight motion-based fallback for minimal environments- Classify gestures: fist, open_palm, pointing, peace_sign, thumbs_upImports reusable modules from `src/` where practical.

In [ ]:
import osimport sysimport numpy as npPROJECT_DIR = "/content/project-aura"if PROJECT_DIR not in sys.path:    sys.path.insert(0, PROJECT_DIR)!pip install --quiet pillowfrom src.core.config import load_configfrom src.agents.gesture import GestureRecognizer, GestureResultfrom src.utils.logging import get_loggerlogger = get_logger("notebook-05")config = load_config()print("Configuration loaded")

In [ ]:
recognizer = GestureRecognizer(    confidence_threshold=config.gesture.confidence_threshold,    max_num_hands=config.gesture.max_num_hands,    min_detection_confidence=config.gesture.min_detection_confidence,    min_tracking_confidence=config.gesture.min_tracking_confidence,)print("Gesture recognizer initialized")print(f"Confidence threshold: {recognizer.confidence_threshold}")print(f"Max hands: {recognizer.max_num_hands}")print(f"MediaPipe available: {recognizer._use_mediapipe}")

In [ ]:
from PIL import Image, ImageDrawdef create_hand_like_image(gesture_type="open_palm", size=(480, 640)):    """Create a synthetic image with hand-like patterns."""    img = Image.new("RGB", size, (50, 50, 50))    draw = ImageDraw.Draw(img)        h, w = size    cx, cy = w // 2, h // 2        if gesture_type == "open_palm":        draw.ellipse([cx-60, cy-40, cx+60, cy+60], fill=(200, 180, 160))        for i in range(5):            fx = cx - 40 + i * 20            draw.rectangle([fx-5, cy-80, fx+5, cy-30], fill=(200, 180, 160))    elif gesture_type == "fist":        draw.ellipse([cx-50, cy-50, cx+50, cy+50], fill=(200, 180, 160))    else:        draw.rectangle([cx-40, cy-40, cx+40, cy+40], fill=(200, 180, 160))        return np.array(img)test_gestures = ["open_palm", "fist", "pointing"]for gesture_name in test_gestures:    img = create_hand_like_image(gesture_name)    results = recognizer.detect(img)        print(f"\nGesture: {gesture_name}")    print(f"  Detections: {len(results)}")    for r in results:        print(f"  - {r.gesture_name} (confidence: {r.confidence:.3f})")

In [ ]:
frame1 = np.zeros((480, 640, 3), dtype=np.uint8)frame2 = np.zeros((480, 640, 3), dtype=np.uint8)frame2[200:280, 300:380] = [255, 255, 255]results1 = recognizer.detect(frame1)results2 = recognizer.detect(frame2)print("Frame 1 (no motion):", len(results1), "detections")print("Frame 2 (with motion):", len(results2), "detections")for r in results2:    print(f"  - {r.gesture_name} (confidence: {r.confidence:.3f})")

In [ ]:
img = create_hand_like_image("open_palm")results = recognizer.detect(img)if results:    result_dict = results[0].to_dict()    print("Serialized gesture result:")    import json    print(json.dumps(result_dict, indent=2, default=str))else:    print("No gestures detected in test image")recognizer.release()print("\nGesture recognizer released")

In [ ]:
print("\n" + "="*60)print("Notebook 05 Complete - Gesture Recognition")print("="*60)print("MediaPipe available:", recognizer._use_mediapipe)print("Fallback mode: motion-based detection")print("Supported gestures: fist, open_palm, pointing, peace_sign, thumbs_up")print("\nNext: Run 06_ai_agents.ipynb for agent orchestration")